In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import friedmanchisquare, rankdata, spearmanr, studentized_range


In [ ]:
DATA_PATH = "results/survey_results_100_participants.csv"

df = pd.read_csv(DATA_PATH)

# Ranking prompts have seven columns such as Q1_1, ..., Q1_7.
# Q19 and Q23 are ease checks, so they are ignored automatically.
question_ids = sorted(
    {
        match.group(1)
        for column in df.columns
        if (match := re.fullmatch(r"(Q\d+)_[1-7]", column))
    },
    key=lambda question: int(question[1:]),
)

assert len(df) == 100, f"Expected 100 participants, found {len(df)}"
assert len(question_ids) == 50, f"Expected 50 ranking prompts, found {len(question_ids)}"

ranking_columns = [
    f"{question}_{position}"
    for question in question_ids
    for position in range(1, 8)
]
rank_data = df[ranking_columns].apply(pd.to_numeric, errors="raise")
assert not rank_data.isna().any().any(), "Ranking data contain missing values"

expected_ranks = np.arange(1, 8)
for question in question_ids:
    columns = [f"{question}_{position}" for position in range(1, 8)]
    observed = np.sort(rank_data[columns].to_numpy(), axis=1)
    if not np.all(observed == expected_ranks):
        invalid_rows = np.flatnonzero(~np.all(observed == expected_ranks, axis=1))
        raise ValueError(
            f"{question} contains invalid rankings in participant rows "
            f"{(invalid_rows + 1).tolist()}"
        )

print(
    f"Validated {len(df)} participants, {len(question_ids)} prompts, "
    "and complete rank permutations from 1 to 7."
)


In [ ]:
# Column suffix corresponding to each model.
model_positions = {
    "DALL-E 2": 1,
    "Mitsua": 2,
    "SD2.1": 3,
    "DreamBooth": 4,
    "LoRA": 5,
    "TI": 6,
    "JuggXL": 7,
}

# One mean rank for each participant--model combination across 50 prompts.
participant_means = pd.DataFrame(
    {
        model: rank_data[
            [f"{question}_{position}" for question in question_ids]
        ].mean(axis=1)
        for model, position in model_positions.items()
    }
)

average_ranks = participant_means.mean().sort_values()
print("Average ranks (lower is better):")
print(average_ranks.round(4))


In [ ]:
# Friedman omnibus test with participants as blocks and models as treatments.
friedman_result = friedmanchisquare(
    *[participant_means[model].to_numpy() for model in participant_means.columns]
)

n_participants, n_models = participant_means.shape
kendalls_w = friedman_result.statistic / (
    n_participants * (n_models - 1)
)

print(
    f"Friedman chi-square({n_models - 1}) = "
    f"{friedman_result.statistic:.6f}"
)
print(f"Friedman p-value = {friedman_result.pvalue:.12g}")
print(f"Kendall's W = {kendalls_w:.6f}")


In [ ]:
# Participant-level cluster bootstrap confidence interval for Kendall's W.
# Participants are resampled as whole blocks, preserving all seven model means.
n_bootstrap = 100_000
bootstrap_seed = 42
batch_size = 5_000

values = participant_means.to_numpy()
within_participant_ranks = np.apply_along_axis(rankdata, 1, values)

# Friedman tie correction associated with each participant block.
tie_penalties = np.zeros(n_participants)
for participant_index, row in enumerate(values):
    _, counts = np.unique(row, return_counts=True)
    tie_penalties[participant_index] = np.sum(counts**3 - counts)

rng = np.random.default_rng(bootstrap_seed)
bootstrap_w = np.empty(n_bootstrap)

for start_index in range(0, n_bootstrap, batch_size):
    current_batch_size = min(batch_size, n_bootstrap - start_index)
    sampled_indices = rng.integers(
        0,
        n_participants,
        size=(current_batch_size, n_participants),
    )

    rank_sums = within_participant_ranks[sampled_indices].sum(axis=1)
    sum_of_squares = np.sum(rank_sums**2, axis=1)
    sampled_ties = tie_penalties[sampled_indices].sum(axis=1)
    tie_correction = 1 - sampled_ties / (
        n_models * (n_models**2 - 1) * n_participants
    )

    bootstrap_statistic = (
        12 / (n_models * n_participants * (n_models + 1))
        * sum_of_squares
        - 3 * n_participants * (n_models + 1)
    ) / tie_correction

    bootstrap_w[start_index : start_index + current_batch_size] = (
        bootstrap_statistic / (n_participants * (n_models - 1))
    )

bootstrap_ci = np.percentile(bootstrap_w, [2.5, 97.5])
print(
    "Participant-bootstrap 95% CI for Kendall's W: "
    f"[{bootstrap_ci[0]:.6f}, {bootstrap_ci[1]:.6f}]"
)


In [ ]:
# Nemenyi post-hoc comparisons using the participant-level observations.
# This implements the Nemenyi-Friedman procedure used by
# scikit_posthocs.posthoc_nemenyi_friedman without requiring that package.
nemenyi_average_ranks = pd.Series(
    within_participant_ranks.mean(axis=0),
    index=participant_means.columns,
)
nemenyi_standard_error = np.sqrt(
    n_models * (n_models + 1) / (6 * n_participants)
)

nemenyi_values = np.ones((n_models, n_models), dtype=float)
for first_index in range(n_models):
    for second_index in range(first_index + 1, n_models):
        rank_difference = abs(
            nemenyi_average_ranks.iloc[first_index]
            - nemenyi_average_ranks.iloc[second_index]
        )
        q_statistic = rank_difference / nemenyi_standard_error
        p_value = studentized_range.sf(
            q_statistic * np.sqrt(2),
            n_models,
            np.inf,
        )
        nemenyi_values[first_index, second_index] = p_value
        nemenyi_values[second_index, first_index] = p_value

nemenyi_result = pd.DataFrame(
    nemenyi_values,
    index=participant_means.columns,
    columns=participant_means.columns,
)

print("Nemenyi post-hoc p-values:")
print(nemenyi_result.round(6))

def format_p_value(value):
    return "p < 0.001" if value < 0.001 else f"p = {value:.3f}"

print(
    "LoRA vs. Mitsua: "
    + format_p_value(nemenyi_result.loc["LoRA", "Mitsua"])
)
print(
    "LoRA vs. SD2.1: "
    + format_p_value(nemenyi_result.loc["LoRA", "SD2.1"])
)


In [ ]:
# Plot the Nemenyi matrix; very small p-values are shown as <0.001.
annotation_labels = np.empty_like(nemenyi_values, dtype=object)
for row_index in range(n_models):
    for column_index in range(n_models):
        value = nemenyi_values[row_index, column_index]
        annotation_labels[row_index, column_index] = (
            "<0.001" if value < 0.001 else f"{value:.3f}"
        )

plt.figure(figsize=(9, 7))
sns.heatmap(
    nemenyi_result,
    annot=annotation_labels,
    fmt="",
    cmap="coolwarm",
    vmin=0,
    vmax=1,
    square=True,
    cbar_kws={"label": "Adjusted p-value"},
)
plt.title("Nemenyi Post-hoc Comparisons")
plt.tight_layout()
plt.savefig("results/nemenyi_posthoc_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# FID-human-preference alignment: correlate each model's mean FID across the
# 76 benchmark categories with its average human preference rank.
FID_DIR = "../4. Scores/results"

fid_scores = pd.read_csv(f"{FID_DIR}/baseline_FID.csv").merge(
    pd.read_csv(f"{FID_DIR}/finetuned_FID.csv"), on="Category"
)
assert len(fid_scores) == 76, f"Expected 76 categories, found {len(fid_scores)}"

fid_columns = {
    "DALL-E 2": "DALLE2_FID",
    "Mitsua": "MITSUA_FID",
    "SD2.1": "SD_FID",
    "DreamBooth": "DREAM_FID",
    "LoRA": "LORA_FID",
    "TI": "TI_FID",
    "JuggXL": "JUGGXL_FID",
}
mean_fid = pd.Series(
    {model: fid_scores[column].mean() for model, column in fid_columns.items()}
)

alignment = pd.DataFrame(
    {
        "Mean FID": mean_fid,
        "FID rank": mean_fid.rank(),
        "Average preference rank": average_ranks,
        "Preference rank": average_ranks.rank(),
    }
).sort_values("Preference rank")

# Lower is better for both FID and preference rank, so a positive rho means agreement.
fid_rho, fid_p = spearmanr(alignment["Mean FID"], alignment["Average preference rank"])

print(alignment.round(3))
print(f"\nFID-preference Spearman rho = {fid_rho:.2f}, p = {fid_p:.4f}")